# Stock Bucket — Volatility Dashboard

Evaluate and compare volatility across a custom basket of stocks:
- **Yang-Zhang volatility** (primary — lowest bias OHLC estimator)
- **Percentile rank** of current vol vs own history
- **Vol z-score** — how many σ above/below the stock's own mean
- **Risk-return scatter** — reward per unit of risk
- **Volatility heatmap** across symbols and time
- **Correlation** of volatility between stocks
- **Regime table** — which stocks are in High / Normal / Low vol right now


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
import talib
from IPython.display import display

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444466',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'text.color':       'white',
    'grid.color':       '#333355',
    'grid.linestyle':   '--',
    'grid.alpha':       0.4,
    'legend.facecolor': '#2a2a4e',
    'legend.labelcolor':'white',
})

ANNUALISE = 252
VOL_WINDOW = 20     # base rolling window for vol calculation
LOOKBACK_DAYS = 504 # ~2 years for percentile ranking
print('Ready')


## 1. Define Your Stock Bucket

In [ ]:
# ── Edit this list with your symbols ─────────────────────────────────────────
BUCKET = [
    'ACB', 'BID', 'CTG', 'MBB', 'STB', 'TCB', 'VCB', 'VPB',  # banks
    'VIC', 'VHM', 'NVL', 'PDR',                                  # real estate
    'HPG', 'HSG', 'NKG',                                          # steel
    'MWG', 'FPT', 'CMG',                                          # tech/retail
    'VNM', 'MSN', 'SAB',                                          # consumer
    'GAS', 'PLX', 'PVD',                                          # energy
    'VNINDEX',                                                     # benchmark
]

# Remove duplicates, keep order
BUCKET = list(dict.fromkeys(BUCKET))
print(f'Bucket: {len(BUCKET)} symbols')
print(BUCKET)


## 2. Load Data

In [ ]:
import os
from deltalake import DeltaTable

REMOTE_HOST = "https://minio.phuchuynh.xyz"
LOCAL_FILE  = "stocks_data_latest.h5"

if os.path.exists(LOCAL_FILE):
    print(f"Loading from local HDF5: {LOCAL_FILE}")
    with pd.HDFStore(LOCAL_FILE, mode='r') as store:
        df_all = store['stocks']
else:
    print("Loading from Delta Lake …")
    storage_options = {
        "AWS_ACCESS_KEY_ID":        "CzOwnLkEDXQy951AOqes",
        "AWS_SECRET_ACCESS_KEY":    "fdRe91TOtqTl0icUkZLsUnWvZa90aZ5qG5rVEf7S",
        "AWS_ENDPOINT_URL":         REMOTE_HOST,
        "AWS_ALLOW_HTTP":           "true",
        "AWS_EC2_METADATA_DISABLED":"true",
        "AWS_REGION":               "us-east-1",
        "aws_conditional_put":      "etag",
    }
    dt = DeltaTable("s3://delta-table-storage/stocks", storage_options=storage_options)
    now = pd.Timestamp.now()
    df_all = dt.to_pandas(
        filters=[("date", ">=", now - pd.DateOffset(years=5)),
                 ("symbol", "in", BUCKET)],
        columns=["symbol", "date", "close", "open", "high", "low", "volume"],
    ).set_index(["date", "symbol"]).unstack(level=1)

# Filter to bucket symbols that exist in data
available = [s for s in BUCKET if s in df_all['close'].columns]
missing    = [s for s in BUCKET if s not in df_all['close'].columns]
if missing:
    print(f"⚠ Not found in data: {missing}")

close  = df_all['close'][available].copy()
open_  = df_all['open'][available].copy()
high   = df_all['high'][available].copy()
low    = df_all['low'][available].copy()

close.index = pd.to_datetime(close.index)
close  = close.sort_index().dropna(how='all')
open_  = open_.reindex(close.index)
high   = high.reindex(close.index)
low    = low.reindex(close.index)

BUCKET = available   # use only what loaded
print(f"Loaded {len(BUCKET)} symbols  |  {close.index[0].date()} → {close.index[-1].date()}")
print(f"Shape: {close.shape}")


## 3. Compute Volatility

In [ ]:
def yang_zhang_vol_df(open_df, high_df, low_df, close_df, window=VOL_WINDOW):
    """Yang-Zhang vol for a DataFrame of symbols (annualised)."""
    log_co = np.log(close_df / open_df)
    log_oc = np.log(open_df / close_df.shift(1))
    rs     = (np.log(high_df / close_df) * np.log(high_df / open_df)
              + np.log(low_df  / close_df) * np.log(low_df  / open_df))
    n   = window
    k   = 0.34 / (1.34 + (n + 1) / (n - 1))
    var = log_oc.rolling(n).var() + k * log_co.rolling(n).var() + (1 - k) * rs.rolling(n).mean()
    return np.sqrt(var.clip(lower=0) * ANNUALISE)


def hist_vol_df(close_df, window=VOL_WINDOW):
    log_ret = np.log(close_df / close_df.shift(1))
    return log_ret.rolling(window).std() * np.sqrt(ANNUALISE)


# Compute
yz_vol = yang_zhang_vol_df(open_, high, low, close)
hv_vol = hist_vol_df(close)

# ── Per-symbol summary metrics ────────────────────────────────────────────────
def vol_stats(vol_df, lookback=LOOKBACK_DAYS):
    """For each symbol: current vol, 1Y mean, percentile rank, z-score, regime."""
    rows = []
    for sym in vol_df.columns:
        s = vol_df[sym].dropna()
        if len(s) < 30:
            continue
        hist  = s.iloc[-lookback:]
        cur   = s.iloc[-1]
        mean  = hist.mean()
        std   = hist.std()
        pctile = (hist < cur).mean() * 100          # percentile rank in own history
        zscore = (cur - mean) / std if std > 0 else 0.0
        if pctile >= 67:   regime = 'High'
        elif pctile <= 33: regime = 'Low'
        else:              regime = 'Normal'
        rows.append({
            'Symbol':     sym,
            'Current %':  round(cur * 100, 2),
            '1Y Mean %':  round(hist.tail(252).mean() * 100, 2),
            'Pctile Rank':round(pctile, 1),
            'Z-Score':    round(zscore, 2),
            'Regime':     regime,
            'Min %':      round(hist.min() * 100, 2),
            'Max %':      round(hist.max() * 100, 2),
        })
    return pd.DataFrame(rows).set_index('Symbol')

stats_df = vol_stats(yz_vol)
print(f"Computed volatility for {len(stats_df)} symbols  (window={VOL_WINDOW}d, YZ estimator)")


## 4. Summary Table

In [ ]:
def style_table(df):
    def color_regime(val):
        colors = {'High': 'background-color:#5d2020; color:white',
                  'Normal': 'background-color:#1a2a3a; color:white',
                  'Low':  'background-color:#1a3a1a; color:white'}
        return colors.get(val, '')

    def color_zscore(val):
        try:
            v = float(val)
            if v >  2: return 'color:#ef9a9a'
            if v >  1: return 'color:#ffb74d'
            if v < -1: return 'color:#a5d6a7'
            return ''
        except: return ''

    return (df.sort_values('Pctile Rank', ascending=False)
              .style
              .applymap(color_regime, subset=['Regime'])
              .applymap(color_zscore, subset=['Z-Score'])
              .background_gradient(subset=['Current %'], cmap='YlOrRd')
              .background_gradient(subset=['Pctile Rank'], cmap='RdYlGn_r')
              .format({'Current %': '{:.2f}%', '1Y Mean %': '{:.2f}%',
                       'Pctile Rank': '{:.1f}', 'Z-Score': '{:+.2f}',
                       'Min %': '{:.2f}%', 'Max %': '{:.2f}%'}))

print(f"As of {close.index[-1].date()} — sorted by Percentile Rank (highest = most elevated vol)")
display(style_table(stats_df))


## 5. Volatility Ranking

In [ ]:
df_sorted = stats_df.sort_values('Current %', ascending=True)
n = len(df_sorted)

# Color by regime
bar_colors = df_sorted['Regime'].map(
    {'High': '#ef9a9a', 'Normal': '#4fc3f7', 'Low': '#a5d6a7'}
)

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, n * 0.35)))

# Left: current vol bar
ax = axes[0]
bars = ax.barh(df_sorted.index, df_sorted['Current %'], color=bar_colors, alpha=0.85, height=0.7)
ax.plot(df_sorted['1Y Mean %'], df_sorted.index, 'o', color='#f7c59f',
        markersize=5, label='1Y Mean', zorder=5)
ax.set_title(f'Current YZ Volatility % (window={VOL_WINDOW}d)', color='white')
ax.set_xlabel('Annualised Vol %')
ax.axvline(df_sorted['Current %'].median(), color='white', lw=1, ls='--', alpha=0.4)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#ef9a9a', label='High'),
                   Patch(color='#4fc3f7', label='Normal'),
                   Patch(color='#a5d6a7', label='Low'),
                   plt.Line2D([0],[0], marker='o', color='#f7c59f', ls='', label='1Y Mean')],
          fontsize=8)
for bar, val in zip(bars, df_sorted['Current %']):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=7, color='white')

# Right: percentile rank
ax2 = axes[1]
pctile_sorted = stats_df.sort_values('Pctile Rank', ascending=True)
pctile_colors = pctile_sorted['Regime'].map(
    {'High': '#ef9a9a', 'Normal': '#4fc3f7', 'Low': '#a5d6a7'}
)
ax2.barh(pctile_sorted.index, pctile_sorted['Pctile Rank'],
         color=pctile_colors, alpha=0.85, height=0.7)
ax2.axvline(50, color='white', lw=1, ls='--', alpha=0.4)
ax2.axvline(67, color='#ef9a9a', lw=1, ls=':', alpha=0.6)
ax2.axvline(33, color='#a5d6a7', lw=1, ls=':', alpha=0.6)
ax2.set_title('Percentile Rank in Own History', color='white')
ax2.set_xlabel('Percentile (%)')
ax2.set_xlim(0, 100)
for i, (sym, row) in enumerate(pctile_sorted.iterrows()):
    ax2.text(row['Pctile Rank'] + 1, i, f'{row["Pctile Rank"]:.0f}',
             va='center', fontsize=7, color='white')

plt.tight_layout()
plt.show()


## 6. Rolling Volatility Over Time

In [ ]:
# Show top-N most volatile + VNINDEX if present
TOP_N = 8
top_syms = stats_df['Current %'].nlargest(TOP_N).index.tolist()
if 'VNINDEX' in BUCKET and 'VNINDEX' not in top_syms:
    top_syms = ['VNINDEX'] + top_syms[:TOP_N-1]

cmap_colors = plt.cm.tab10(np.linspace(0, 1, len(top_syms)))

fig, ax = plt.subplots(figsize=(15, 6))
for sym, c in zip(top_syms, cmap_colors):
    s = yz_vol[sym].dropna()
    lw = 2.0 if sym == 'VNINDEX' else 0.9
    ax.plot(s.index, s * 100, lw=lw, label=sym, color=c, alpha=0.9)

ax.set_title(f'Yang-Zhang Volatility — Top {TOP_N} Most Volatile + VNINDEX', color='white')
ax.set_ylabel('Annualised Vol %')
ax.legend(ncol=4, fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()


## 7. Volatility Heatmap — Symbols × Time

In [ ]:
# Monthly average vol per symbol
vol_monthly = yz_vol.copy()
vol_monthly.index = pd.to_datetime(vol_monthly.index)

pivot = (vol_monthly
         .groupby(vol_monthly.index.to_period('M'))
         .mean() * 100)

# Keep last 24 months
pivot = pivot.iloc[-24:]
pivot.index = [str(p) for p in pivot.index]

# Sort symbols by avg vol descending
sym_order = pivot.mean().sort_values(ascending=False).index.tolist()
pivot = pivot[sym_order]

fig, ax = plt.subplots(figsize=(max(12, len(pivot.columns) * 0.55),
                                max(6,  len(pivot.index) * 0.45)))

cmap = LinearSegmentedColormap.from_list('vol', ['#1a237e', '#4fc3f7', '#ffb74d', '#ef9a9a'])
im = ax.imshow(pivot.T.values, aspect='auto', cmap=cmap)

ax.set_xticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.index, rotation=45, ha='right', fontsize=7, color='white')
ax.set_yticks(range(len(pivot.columns)))
ax.set_yticklabels(pivot.columns, fontsize=8, color='white')
ax.set_title('Monthly Average YZ Volatility (%) — last 24 months', color='white', fontsize=12)
plt.colorbar(im, ax=ax, label='Annualised Vol %', shrink=0.6)

# Annotate cells
vmin, vmax = pivot.values[~np.isnan(pivot.values)].min(), pivot.values[~np.isnan(pivot.values)].max()
for j, sym in enumerate(pivot.columns):
    for i, month in enumerate(pivot.index):
        v = pivot.loc[month, sym]
        if not np.isnan(v):
            text_color = 'white' if v < (vmin + vmax) / 2 else '#1a1a2e'
            ax.text(i, j, f'{v:.0f}', ha='center', va='center',
                    fontsize=6, color=text_color)

plt.tight_layout()
plt.show()


## 8. Risk-Return Scatter

In [ ]:
log_ret = np.log(close / close.shift(1))
ann_ret = log_ret.tail(252).mean() * ANNUALISE * 100
ann_vol = yz_vol.tail(252).mean() * 100

fig, ax = plt.subplots(figsize=(11, 7))

regime_color = stats_df['Regime'].map(
    {'High': '#ef9a9a', 'Normal': '#4fc3f7', 'Low': '#a5d6a7'}
)

for sym in ann_vol.index:
    if sym not in ann_ret.index: continue
    x, y = ann_vol[sym], ann_ret[sym]
    if np.isnan(x) or np.isnan(y): continue
    color = regime_color.get(sym, '#4fc3f7')
    ax.scatter(x, y, color=color, s=70, alpha=0.85, zorder=3)
    ax.annotate(sym, (x, y), textcoords='offset points', xytext=(5, 3),
                fontsize=7, color='white', alpha=0.9)

# Iso-Sharpe lines
x_range = np.linspace(ann_vol.min() * 0.8, ann_vol.max() * 1.1, 200)
for sharpe, ls in [(0.5, ':'), (1.0, '--'), (2.0, '-.')]:
    ax.plot(x_range, sharpe * x_range, lw=0.8, ls=ls, color='#888899',
            label=f'Sharpe = {sharpe}')

ax.axhline(0, color='#666688', lw=0.8)
ax.axvline(ann_vol.median(), color='#666688', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('1Y Average Volatility % (YZ)')
ax.set_ylabel('1Y Annualised Return %')
ax.set_title('Risk-Return: 1-Year Lookback', color='white', fontsize=12)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#ef9a9a', label='High Vol Regime'),
    Patch(color='#4fc3f7', label='Normal Vol Regime'),
    Patch(color='#a5d6a7', label='Low Vol Regime'),
] + [plt.Line2D([0],[0], color='#888899', ls=ls, label=f'Sharpe={s}')
     for s, ls in [(0.5,':'),(1.0,'--'),(2.0,'-.')]],
    fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## 9. Volatility Correlation Between Stocks

In [ ]:
# Pearson correlation of rolling vol time-series
vol_corr = yz_vol.tail(504).corr()

fig, ax = plt.subplots(figsize=(max(8, len(BUCKET) * 0.55),
                                max(7, len(BUCKET) * 0.5)))

norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
cmap2 = LinearSegmentedColormap.from_list('corr', ['#ef9a9a', '#1a1a2e', '#4fc3f7'])
im = ax.imshow(vol_corr.values, cmap=cmap2, norm=norm, aspect='auto')

ticks = range(len(vol_corr))
ax.set_xticks(ticks); ax.set_xticklabels(vol_corr.columns, rotation=45, ha='right', fontsize=7, color='white')
ax.set_yticks(ticks); ax.set_yticklabels(vol_corr.index,   fontsize=7,  color='white')
ax.set_title('Volatility Correlation Matrix (2Y)', color='white', fontsize=12)
plt.colorbar(im, ax=ax, label='Pearson r', shrink=0.7)

for i in range(len(vol_corr)):
    for j in range(len(vol_corr)):
        v = vol_corr.iloc[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                fontsize=5.5, color='white' if abs(v) < 0.6 else '#1a1a2e')

plt.tight_layout()
plt.show()


## 10. Volatility Z-Score (Current vs Own History)

In [ ]:
zscores = stats_df['Z-Score'].sort_values(ascending=True)
colors  = ['#ef9a9a' if z > 1 else '#a5d6a7' if z < -1 else '#4fc3f7'
           for z in zscores]

fig, ax = plt.subplots(figsize=(12, max(5, len(zscores) * 0.35)))
bars = ax.barh(zscores.index, zscores.values, color=colors, alpha=0.85, height=0.7)
ax.axvline(0,  color='white',   lw=1.2)
ax.axvline( 1, color='#ef9a9a', lw=1, ls='--', alpha=0.7, label='+1σ')
ax.axvline(-1, color='#a5d6a7', lw=1, ls='--', alpha=0.7, label='-1σ')
ax.axvline( 2, color='#ef5350', lw=1, ls=':',  alpha=0.6, label='+2σ')
ax.axvline(-2, color='#66bb6a', lw=1, ls=':',  alpha=0.6, label='-2σ')
for bar, val in zip(bars, zscores):
    ax.text(val + 0.05 if val >= 0 else val - 0.05,
            bar.get_y() + bar.get_height() / 2,
            f'{val:+.2f}σ', va='center', ha='left' if val >= 0 else 'right',
            fontsize=7, color='white')
ax.set_title('Vol Z-Score: how many σ is current vol from its own 2Y mean', color='white')
ax.set_xlabel('Standard Deviations from Mean')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 11. Regime Summary

In [ ]:
for regime in ['High', 'Normal', 'Low']:
    syms = stats_df[stats_df['Regime'] == regime].sort_values('Pctile Rank', ascending=False)
    icon = {'High': '🔴', 'Normal': '🔵', 'Low': '🟢'}.get(regime, '')
    print(f"\n{icon} {regime} Volatility ({len(syms)} stocks)")
    if len(syms):
        print(syms[['Current %','1Y Mean %','Pctile Rank','Z-Score']].to_string())
